# 2. Vectorise: the same maths, in compiled C

The Python loop was slow not because the *arithmetic* is hard - it is nine multiplies -
but because the interpreter does that arithmetic one operation at a time, with loop and
type-dispatch overhead on every single one.

**NumPy** removes the interpreter from the inner loop. Instead of `for each pixel: for
each tap`, you express the whole convolution as a handful of whole-array operations
(`acc += coeff[k] * shifted_image`), each of which runs as one tight compiled C loop -
often using the CPU's SIMD units to do several pixels per instruction. The loop is still
there; it just isn't in Python any more. This is the `software` backend, and it is our
golden reference (`conv_reference`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

In [ ]:
# Same operation, same correctness check - now vectorised, on a much bigger image than
# the Python loop could stomach.
img = sample_gray(256)
out, t = run_rung('software', img, 'edges', repeats=20, scoreboard=sb)

That is a speedup of many orders of magnitude over the triple loop - for *identical*
output. The lesson: most of the "FPGA vs CPU" gap people imagine is really "good code vs
bad code". Beat the easy wins first.

But NumPy has a ceiling. A CPU core still executes a bounded number of arithmetic
operations per clock, and for each output pixel it re-reads the neighbouring pixels from
memory. To go faster we need to do all nine multiplies *at once*, every clock, and stop
re-reading data - which is what hardware lets us build.

In [ ]:
# Where we are now: two software rungs on the roofline, and the running scoreboard.
sb.roofline(); plt.show()
sb.progress(); plt.show()

> Note: on a laptop, NumPy may sit *above* the 100 MHz PL ceiling - a modern laptop core
> is faster than our small fabric. The honest comparison is same-system: run these
> software rungs on the *board's* ARM cores (the PS) against the accelerator in the PL.
> Then the climb is apples-to-apples.

**Next:** what an FPGA actually is, and our first circuit - written by hand in Verilog.
-> `03_rtl_fpga.ipynb`.